In [ ]:
# Colab setup (added for Google Colab): run this cell first. It does nothing outside Colab.
# It fetches this practical's data/ folder from GitHub, so the notebook runs unchanged.
import os, shutil, subprocess, sys
from pathlib import Path

REPO = "https://github.com/TAHMO/NOAA.git"
BRANCH = "zambia-workshop-2"
FOLDER = "Exercise_3/SPW_v2_lite_practical"

if ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ) and not Path("data").is_dir():
    clone = Path("/content/NOAA")
    if not (clone / FOLDER / "data").is_dir():
        shutil.rmtree(clone, ignore_errors=True)
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                        "-b", BRANCH, REPO, str(clone)], check=True)
        subprocess.run(["git", "-C", str(clone), "sparse-checkout", "set", FOLDER + "/data"], check=True)
    os.chdir(clone / FOLDER)
    print("Data ready:", Path("data").resolve())


# Short-term forecasts and planting decisions
## SPW notebook toolkit

**SPW** (Suitable Planting Window) tells farmers whether enough rain is likely to plant. This practical runs
**SPW v2**, the rain rules GAP uses with farmers (as of November 2026), at one example point in Zambia, Malawi or Nigeria for
the 2023 and 2024 seasons, and checks it against what really happened: the rain, the soil and farmers' yields.

### How SPW v2 decides

| Step | What SPW v2 does |
|---|---|
| 1. Add up the rain | The rain **already fallen** in the last 6 days + the rain the **forecast** expects in the next 5 days |
| 2. Ask how sure | The forecast has many members. If the chance that the 11-day total is above **30 mm** is at least **80 %**: *rain test passed*. If not, but the median of the members is 30 mm or more: *plant with caution*. Otherwise: *wait* |
| 3. Check the calendar | SPW only advises inside the **planting window** from the season calendar. Before it, the answer is *prepare* |

The full GAP SPW also checks soil moisture and long-term rainfall; this notebook (**SPW v2 lite**) uses the rain rules only.
For comparison, the notebook also shows the **original rule** from the earlier toolkit (40 mm).

### How to use this notebook

1. **Run the setup cell** in Section 1 once, then **choose the country and season once**. Every section uses that
   choice. To switch, change it there and use **Run → Run All Cells**.
2. Each section follows the same pattern: 🎯 an **objective** box; a **settings** cell where there is something to
   change; the **chart** and **tables** (code folded away: click the grey bar to see it); a **key result** box that
   reads the numbers back in plain words; 💬 **questions to discuss** and one ✍️ **key question**.
3. **After changing a setting, run every cell below it in that section**, so the key-result box matches the chart.
4. Section 7 brings together **what we learned**. **Section 8 is your group output**: your answers to the five key
   questions and your planting window.

| Section | What you will learn | Task |
|---|---|---|
| 1 | Where your example point is | Task 1 |
| 2 | One rain check: would SPW v2 have said plant on this day? | Task 2 · key question 1 |
| 3 | Through the season: when did SPW v2 first say plant? | Task 3 · key question 2 |
| 4 | How good was the forecast part of the rain total? | Task 4 · key question 3 |
| 5 | What was the soil like after SPW said plant? | Task 5 · key question 4 |
| 6 | Malawi farms: did planting in line with SPW go with better yields? (everyone) | Task 6 · key question 5 |
| 7 | What we learned | — |
| 8 | **Your answers and planting window**, presented to the room | Group output · everyone |

**Scope:** one example grid point per country (Chipata area, Lilongwe area, Kaduna area), not a country average.
Rainfall: TAMSAT. Forecasts: ECMWF reforecasts (2025 archive). Soil moisture and farm yields are used only to check
outcomes afterwards, never as inputs to the decision.

### 1. Setup and your example location

> 🎯 **Objective:** get the notebook running and find your example point.
>
> **You will learn to:**
> - choose the country and season used by every section
> - see that the data describe one grid point, not a country average
>
> **Task sheet:** Task 1

**In short:** Run the setup cell once. It loads the tools and shows nothing except a short confirmation. Python packages needed: numpy, pandas, matplotlib, jinja2 and jupyter.

In [ ]:
# Run this setup cell once.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path("data")
plt.rcParams.update({"font.size": 11, "figure.figsize": (11, 5)})

def load_country(country):
    """Load the point-example data afresh for the country selected for this plot."""
    locations = pd.read_csv(DATA_DIR / "locations.csv")
    if country not in locations.country.values:
        raise ValueError("Choose Zambia, Malawi or Nigeria.")
    site = locations.loc[locations.country.eq(country)].iloc[0]
    slug = country.lower()
    obs = pd.read_csv(DATA_DIR / f"{slug}_observed_rainfall.csv", parse_dates=["date"])
    forecasts = pd.read_csv(DATA_DIR / f"{slug}_forecast_daily.csv", parse_dates=["initialization_date", "date"])
    spw = pd.read_csv(DATA_DIR / f"{slug}_spw_daily.csv", parse_dates=["date"])
    member_columns = [c for c in forecasts if c.startswith("member_")]
    return site, slug, obs, forecasts, spw, member_columns

def observed_days(start, end):
    values = obs.set_index("date").rainfall_mm.reindex(pd.date_range(start, end, freq="D"))
    if values.isna().any():
        raise ValueError("The requested observation window is incomplete.")
    return values

def rainfall_window(initialization, threshold_mm, past_days=5, future_days=6):
    initialization = pd.Timestamp(initialization)
    if threshold_mm <= 0 or not 1 <= future_days <= 15 or past_days < 1:
        raise ValueError("Use a positive threshold, at least one past day and 1–15 forecast days.")
    forecast = forecasts.loc[forecasts.initialization_date.eq(initialization)].sort_values("date").head(future_days)
    expected = pd.date_range(initialization, periods=future_days, freq="D")
    if len(forecast) != future_days or not np.array_equal(forecast.date.values, expected.values):
        raise ValueError("Choose a supplied initialisation with a complete forecast window.")
    past = observed_days(initialization-pd.Timedelta(days=past_days), initialization-pd.Timedelta(days=1))
    future = forecast[member_columns].to_numpy(dtype=float)
    if not np.isfinite(future).all():
        raise ValueError("Missing forecast member values.")
    totals = past.sum() + future.sum(axis=0)
    observed_future = observed_days(initialization, expected[-1])
    return past, forecast, totals, observed_future


# ---------------------------------------------------------------------------
# SPW v2 (GAP) — rainfall components. Values mirror the GAP configuration:
# thresholds skills/indices/spw/thresholds/{ZM,MW}; windows skills/seasons (maize).
# ---------------------------------------------------------------------------
import math

# advice_floor = first date farmers may be advised; watch_date = earlier "start watching" date where the
# calendar has one. SPW starts assessing on whichever is earlier (GAP spw_open.spw_window_start).
SPW_V2 = {
    "Zambia": {"calendar": "Chipata (maize)", "advice_floor": "11-19", "watch_date": "11-14", "window_end": "01-21"},
    "Malawi": {"calendar": "Lilongwe (maize)", "advice_floor": "11-21", "watch_date": None, "window_end": "01-07"},
    "Nigeria": {"calendar": "Guinea Savanna zone incl. Kaduna (maize)", "advice_floor": "05-01", "watch_date": None,
                "window_end": "07-09"},
}
V2_PAST_DAYS, V2_FUTURE_DAYS = 6, 5   # observed days before + forecast days from the advisory date
V2_THRESHOLD_MM = 30.0                 # in-window threshold (Zambia, Malawi and Nigeria all use 30 mm)
V2_P_SHORT = 0.80                      # probability needed (conservative mode)
V2_ASSESSMENT_LEAD_DAYS = 0            # GAP default: no extra lead before the advice date
V2_BEFORE_BUFFER_DAYS = 60             # "before the window" lasts this long
V2_MAX_SIGMA_RATIO = 3.0
V2_LATE_PAST_DAYS, V2_LATE_FUTURE_DAYS = 1, 3   # after the window: 1 observed + 3 forecast days
V2_LATE_THRESHOLD_MM, V2_P_LATE = 5.0, 0.50

def prob_exceed_normal(mu, sigma, threshold, max_sigma_ratio=V2_MAX_SIGMA_RATIO):
    """P(total > threshold) from a Normal curve, as in GAP's spw_probability.prob_exceed_normal."""
    if not (np.isfinite(mu) and np.isfinite(sigma)):
        return np.nan
    if sigma > 0 and mu > 0:
        sigma = min(sigma, max_sigma_ratio * (max(mu, 0.0) + 1.0))
    if sigma <= 0 or mu <= 0:
        return 1.0 if mu > threshold else 0.0
    return float(np.clip(0.5 * math.erfc((threshold - mu) / (sigma * math.sqrt(2.0))), 0.0, 1.0))

def season_window(country, year):
    """(assessment opens, planting window closes) for the season that starts in `year`."""
    cfg = SPW_V2[country]
    floor = pd.Timestamp(f"{year}-{cfg['advice_floor']}")
    opens = floor - pd.Timedelta(days=V2_ASSESSMENT_LEAD_DAYS)
    if cfg["watch_date"]:
        opens = min(opens, pd.Timestamp(f"{year}-{cfg['watch_date']}"))
    closes = pd.Timestamp(f"{year}-{cfg['window_end']}")
    if closes < floor:                         # window runs across the new year
        closes = pd.Timestamp(f"{year + 1}-{cfg['window_end']}")
    return opens, closes

def spw_phase(advisory_date, country):
    """TOO_EARLY / BEFORE / IN / AFTER, as GAP's classify_spw_phase."""
    day = pd.Timestamp(advisory_date)
    buffer = pd.Timedelta(days=V2_BEFORE_BUFFER_DAYS)
    seasons = [season_window(country, year) for year in (day.year - 1, day.year, day.year + 1)]
    if any(opens <= day <= closes for opens, closes in seasons):
        return "IN"
    if any(closes < day <= closes + buffer for opens, closes in seasons):
        return "AFTER"
    if any(opens - buffer <= day < opens for opens, closes in seasons):
        return "BEFORE"
    return "TOO_EARLY"

def spw_v2_day(advisory_date, forecast_init, country, support):
    """The v2 rainfall decision for one advisory date, using the latest available forecast."""
    day, init = pd.Timestamp(advisory_date), pd.Timestamp(forecast_init)
    phase = spw_phase(day, country)
    row = {"date": day, "phase": phase, "forecast_init": init, "forecast_age_days": (day - init).days,
           "observed_mm": np.nan, "mean_mm": np.nan, "median_mm": np.nan, "sigma_mm": np.nan,
           "p10_mm": np.nan, "p90_mm": np.nan, "probability": np.nan, "late_probability": np.nan}
    forecast = support.loc[support.initialization_date.eq(init)].set_index("date")[member_columns]

    def spliced_totals(past_days, future_days):
        """Observed total + each member's forecast total, or None if any needed value is missing."""
        try:
            observed = observed_days(day - pd.Timedelta(days=past_days), day - pd.Timedelta(days=1)).sum()
        except ValueError:
            return None, None
        members = forecast.reindex(pd.date_range(day, periods=future_days)).to_numpy(dtype=float)
        if not np.isfinite(members).all():
            return None, None
        return observed, observed + members.sum(axis=0)

    # The 11-day window: used in the planting window, and shown as a diagnostic in every phase.
    observed, totals = spliced_totals(V2_PAST_DAYS, V2_FUTURE_DAYS)
    if totals is not None:
        row.update(observed_mm=observed, mean_mm=totals.mean(), median_mm=float(np.percentile(totals, 50)),
                   sigma_mm=float(np.std(totals)), p10_mm=float(np.percentile(totals, 10)),
                   p90_mm=float(np.percentile(totals, 90)))
        row["probability"] = prob_exceed_normal(row["mean_mm"], row["sigma_mm"], V2_THRESHOLD_MM)
    if phase == "IN":
        if totals is None:
            return {**row, "state": "NO_DATA"}
        state = ("RAIN_OK" if row["probability"] >= V2_P_SHORT
                 else "PLANT_CAUTION" if row["median_mm"] >= V2_THRESHOLD_MM else "WAIT")
    elif phase == "AFTER":
        # After the window GAP uses only the short late-planting window.
        _, late_totals = spliced_totals(V2_LATE_PAST_DAYS, V2_LATE_FUTURE_DAYS)
        if late_totals is None:
            return {**row, "state": "NO_DATA"}
        row["late_probability"] = prob_exceed_normal(late_totals.mean(), float(np.std(late_totals)),
                                                     V2_LATE_THRESHOLD_MM)
        state = "LATE_OK" if row["late_probability"] >= V2_P_LATE else "LATE_NO"
    elif phase == "BEFORE":
        state = "PREPARE"
    else:
        state = phase
    return {**row, "state": state}


# ---- Display helpers for results: coloured tables and key-result boxes ----------------------
from IPython.display import HTML

def key_result(title, lines, colour="#367a9c"):
    """Show the main finding in a coloured box above a table."""
    items = "".join(f"<li style='margin:4px 0'>{line}</li>" for line in lines)
    display(HTML(f"<div style='border-left:6px solid {colour};background:#f4f8fa;color:#1d2b33;"
                 f"padding:10px 16px;margin:14px 0 8px 0;border-radius:4px;font-size:14px'>"
                 f"<b style='font-size:15px'>{title}</b><ul style='margin:6px 0 0 0;padding-left:20px'>"
                 f"{items}</ul></div>"))

def results_table(shown, caption, difference=None, reference_row=None, label_colours=None, bar_column=None):
    """Style a results table. `shown` holds the text to display; `difference` (same index) holds the
    numeric % differences used to colour that column: green = higher yield, red = lower."""
    def colour_difference(column):
        out = []
        for i in column.index:
            d = None if difference is None else difference.get(i)
            if d is None or pd.isna(d):
                out.append("")
            else:
                strength = min(abs(d) / 20, 1) * .55 + .12
                rgb = "46,139,87" if d > 0 else "205,80,60"
                out.append(f"background-color:rgba({rgb},{strength:.2f});font-weight:600")
        return out
    def colour_rows(row):
        styles = [""] * len(row)
        if reference_row is not None and row.name == reference_row:
            styles = ["background-color:#e3eef4;font-weight:600"] * len(row)
        if label_colours and row.iloc[0] in label_colours:
            styles[0] += f";border-left:8px solid {label_colours[row.iloc[0]]};font-weight:600"
        return styles
    styler = shown.style.hide(axis="index").apply(colour_rows, axis=1).set_caption(caption)
    if difference is not None and "Difference" in shown:
        styler = styler.apply(colour_difference, subset=["Difference"])
    if bar_column:
        styler = styler.bar(subset=[bar_column], color="#cfe3ec", vmin=0)
    return styler.set_table_styles([
        {"selector": "", "props": "border-collapse:collapse;margin:6px 0 22px 0;font-size:14px;"
                                   "background:white;color:#1d2b33"},
        {"selector": "caption", "props": "caption-side:top;text-align:left;font-size:15px;font-weight:600;"
                                          "color:#1d2b33;padding:4px 0 8px 0"},
        {"selector": "th", "props": "background:#2f5d73;color:white;padding:8px 14px;text-align:left"},
        {"selector": "td", "props": "padding:8px 14px;border-bottom:1px solid #dde5ea;color:#1d2b33;text-align:left"},
    ])

# ---- Season helpers used by Sections 3, 4, 5 and 7 -------------------------------------------
STATE_STYLE = {"TOO_EARLY": ("Too early", "#eeeeee"), "PREPARE": ("Before the window: prepare", "#cfd8dc"),
               "WAIT": ("Wait", "#d98d64"), "PLANT_CAUTION": ("Plant with caution", "#f2c14e"),
               "RAIN_OK": ("Rain test passed", "#367a9c"), "LATE_OK": ("After the window: late planting may work", "#79afce"),
               "LATE_NO": ("After the window: late planting not advised", "#9e9e9e"), "NO_DATA": ("No forecast", "#ffffff")}
SOIL_THRESHOLD = {"Zambia": 45, "Malawi": 45, "Nigeria": 70}   # GAP SPW v2.5, % of field capacity (not used by SPW v2 lite)


def use_country(country):
    """Load a country's data and make it the data every helper uses."""
    global site, slug, obs, forecasts, spw, member_columns
    site, slug, obs, forecasts, spw, member_columns = load_country(country)


def season_dates(country, year):
    example = pd.read_csv(DATA_DIR / "locations.csv").set_index("country").loc[country]
    return pd.Timestamp(f"{year}{example.window_start[4:]}"), pd.Timestamp(f"{year}{example.window_end[4:]}")


def season_v2(country, year):
    """SPW v2 answer for every day of the supplied season, with the original 40 mm flag and the soil outcome."""
    use_country(country)
    start, end = season_dates(country, year)
    calendar = spw.set_index("date").loc[start:end]
    support = pd.read_csv(DATA_DIR / f"{slug}_forecast_support.csv", parse_dates=["initialization_date", "date"])
    v2 = pd.DataFrame([spw_v2_day(day, record.forecast_initialization_date, country, support)
                       for day, record in calendar.iterrows()]).set_index("date")
    v2["original_40"] = calendar.spw_40.eq(1)
    v2["soil_next_15_days"] = calendar.postplant_pfc_15days
    return v2


def first_day(mask):
    return mask.index[mask].min() if mask.any() else pd.NaT


def show_day(day):
    return f"{day:%d %b %Y}" if pd.notna(day) else "none this season"


def splice_check(country, year, past_days=V2_PAST_DAYS, future_days=V2_FUTURE_DAYS):
    """For every forecast date in the season: SPW's rain total (rain fallen + forecast) against the rain that fell."""
    use_country(country)
    start, end = season_dates(country, year)
    inits = forecasts.initialization_date[forecasts.initialization_date.between(start, end)].drop_duplicates().sort_values()
    rows = []
    for init in inits:
        recent, case_forecast, members, observed_future = rainfall_window(init, 1, past_days, future_days)
        rows.append({"initialization_date": init, "window_start": recent.index[0], "window_end": case_forecast.date.iloc[-1],
                     "recent_observed_mm": recent.sum(), "observed_total_mm": recent.sum() + observed_future.sum(),
                     "spliced_ensemble_mean_mm": members.mean(),
                     "spliced_p10_mm": np.quantile(members, .1), "spliced_p90_mm": np.quantile(members, .9)})
    return pd.DataFrame(rows)


print("Setup complete. Now choose the country and season in the cell below.")

#### Choose your country and season

**Choose once here**: Zambia (Chipata area), Malawi (Lilongwe area) or Nigeria (Kaduna area), and 2023 or 2024.
Teams from other countries: choose the example closest to your own country.

In [ ]:
# ✏️ Choose the country and season ONCE — every section below follows this choice
COUNTRY = "Zambia"   # Zambia, Malawi or Nigeria
YEAR = 2023          # 2023 or 2024 (start with 2023)

if COUNTRY not in SPW_V2 or YEAR not in (2023, 2024):
    raise ValueError("Choose Zambia, Malawi or Nigeria, and 2023 or 2024.")
_example = pd.read_csv(DATA_DIR / "locations.csv").set_index("country").loc[COUNTRY]
SEASON_START = f"{YEAR}{_example.window_start[4:]}"         # Zambia/Malawi: Sep–Dec; Nigeria: Apr–Jul
SEASON_END = f"{YEAR}{_example.window_end[4:]}"
EXAMPLE_DATE = f"{YEAR}{_example.example_initialization[4:]}"
print(f"{COUNTRY} {YEAR}: season {SEASON_START} to {SEASON_END}; example forecast date {EXAMPLE_DATE}")

#### Your example point

In [ ]:
site, slug, obs, forecasts, spw, member_columns = load_country(COUNTRY)

print(f"{COUNTRY}: {site.site_name}")
point_table = pd.DataFrame({"Item": ["Country", "Example area", "Point asked for (latitude, longitude)",
                                     "SPW grid point", "Rainfall grid point", "Forecast grid point"],
                            "Value": [COUNTRY, site.site_name,
                                      f"{site.requested_latitude}, {site.requested_longitude}",
                                      f"{site.spw_latitude}, {site.spw_longitude}",
                                      f"{site.rainfall_latitude}, {site.rainfall_longitude}",
                                      f"{site.forecast_latitude}, {site.forecast_longitude}"]})
display(results_table(point_table,
                      "Example location"))
# Location map — uses bundled boundaries; no extra packages or downloads.
import json
from matplotlib.patches import Polygon

with (DATA_DIR / "africa_country_boundaries.geojson").open() as boundary_file:
    boundaries = json.load(boundary_file)["features"]
selected_country = next(f for f in boundaries if f["properties"]["ADMIN"] == COUNTRY)

def polygon_rings(feature):
    geometry = feature["geometry"]
    polygons = [geometry["coordinates"]] if geometry["type"] == "Polygon" else geometry["coordinates"]
    return [np.asarray(polygon[0]) for polygon in polygons]

country_points = np.vstack(polygon_rings(selected_country))
west, south = country_points.min(axis=0)
east, north = country_points.max(axis=0)
padding = max(east-west, north-south)*.15
point_lon, point_lat = site.rainfall_longitude, site.rainfall_latitude
fig, axes = plt.subplots(1, 2, figsize=(12, 6), gridspec_kw={"width_ratios": [1, 1.35]})
for ax in axes:
    ax.set_facecolor("#f0f6f8")
    for feature in boundaries:
        is_selected = feature["properties"]["ADMIN"] == COUNTRY
        for ring in polygon_rings(feature):
            ax.add_patch(Polygon(ring, closed=True, facecolor="#d9e7cf" if is_selected else "#f5f4ef",
                                 edgecolor="#777777", linewidth=.65))
    ax.scatter(point_lon, point_lat, s=75, color="#b83b35", edgecolors="white", linewidth=.9,
               zorder=5, label="Example grid point")
    ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
    ax.grid(alpha=.15); ax.set_axisbelow(True)
axes[0].set(xlim=(-20, 55), ylim=(-36, 38), title="Location within Africa")
axes[0].set_aspect("equal")
axes[1].set(xlim=(west-padding, east+padding), ylim=(south-padding, north+padding), title=COUNTRY)
axes[1].set_aspect(1/np.cos(np.deg2rad(point_lat)))
axes[1].annotate(site.site_name, (point_lon, point_lat), xytext=(-12, 18),
                 textcoords="offset points", ha="right", fontsize=10,
                 bbox={"facecolor":"white", "edgecolor":"none", "alpha":.85},
                 arrowprops={"arrowstyle":"-", "color":"#555555"})
axes[1].legend(loc="lower left", fontsize=9)
fig.suptitle(f"{COUNTRY} | {site.site_name} — POINT EXAMPLE, NOT A COUNTRY AVERAGE", fontsize=13)
fig.text(.5, .04, f"Data location: {point_lat:.3f}° latitude, {point_lon:.3f}° longitude. "
         "Rainfall grid point; forecasts interpolated to the same point.", ha="center", fontsize=9)
fig.text(.5, .01, "Country outlines: Natural Earth 1:110m (public domain); indicative boundaries.", ha="center", fontsize=8, color="#555555")
plt.tight_layout(rect=(0, .08, 1, .93)); plt.show()

> 💬 **Questions to discuss — Task 1**
>
> 1. Where is your example point? Is it typical of where maize is grown in your country?
> 2. Why does one grid point not tell you about the whole country?

### 2. One rain check: would SPW v2 have said plant on this day?

> 🎯 **Objective:** see how SPW v2 combines the rain already fallen with the forecast, for one forecast date.
>
> **You will learn to:**
> - split the 11-day rain total into rain already fallen and rain forecast
> - read the forecast members as a range of possible outcomes, not one number
> - turn the total into an SPW v2 answer: rain test passed, plant with caution or wait
>
> **Task sheet:** Task 2 · key question 1

**In short:** SPW adds the rain of the **last 6 days** to the rain the forecast expects in the **next 5 days**. The forecast is 10 possible outcomes (*members*); if the chance that the total is above 30 mm is at least 80 %, the rain test is passed.

<details><summary><b>More detail</b> (click to open)</summary>

The last observed day is the day before the forecast date. The same observed total is added to every member.
The chance of more than 30 mm comes from a Normal curve fitted to the members' totals (their mean and spread), as in
GAP, so it can differ slightly from the share of members above 30 mm. Rainfall: TAMSAT at the nearest 0.25° grid point;
forecasts: ECMWF, interpolated to the same point. This is one rain check on one day; Section 3 runs it for every day.

</details>

In [ ]:
# ✏️ Settings: the forecast date to check (the dates you can choose are listed under the chart)
ISSUE_DATE = EXAMPLE_DATE   # or type one of the dates listed under the chart, in quotes, e.g. "2023-11-27"

#### A. The rain window: what fell, and what the forecast expected

Top: daily rain (blue = observed, orange = forecast average and 10–90 % range). Middle: the running total against 30 mm. Bottom: the 11-day total for each forecast member.

In [ ]:
use_country(COUNTRY)
RAINFALL_THRESHOLD_MM, PAST_DAYS, FUTURE_DAYS = int(V2_THRESHOLD_MM), V2_PAST_DAYS, V2_FUTURE_DAYS

# Only initialisations included in the supplied data can be selected.
available_dates = sorted(forecasts.initialization_date.dt.strftime("%Y-%m-%d").unique())
try:
    parsed_date = pd.to_datetime(ISSUE_DATE, format="%Y-%m-%d", errors="raise")
    if not isinstance(ISSUE_DATE, str) or parsed_date.strftime("%Y-%m-%d") != ISSUE_DATE:
        raise ValueError
except (ValueError, TypeError):
    raise ValueError('Set ISSUE_DATE to a date in quotes using YYYY-MM-DD, e.g. "2023-11-23".') from None
season_choices = [d for d in available_dates if SEASON_START <= d <= SEASON_END]
if ISSUE_DATE not in season_choices:
    wanted = pd.Timestamp(ISSUE_DATE)
    if pd.Timestamp(SEASON_START) <= wanted <= pd.Timestamp(SEASON_END):
        chosen = min(season_choices, key=lambda d: abs(pd.Timestamp(d) - wanted))
        print(f"⚠️ No forecast on {ISSUE_DATE}; using the nearest forecast date, {chosen}.")
    else:
        chosen = EXAMPLE_DATE
        print(f"⚠️ {ISSUE_DATE} is not in the {YEAR} season; using the example date {chosen}. "
              "To check another day, type one of the dates listed under the chart.")
    ISSUE_DATE = chosen

past, forecast, totals, observed_future = rainfall_window(
    ISSUE_DATE, RAINFALL_THRESHOLD_MM, PAST_DAYS, FUTURE_DAYS)
issue = pd.Timestamp(ISSUE_DATE)
ensemble = forecast[member_columns].to_numpy(dtype=float)
fig, axes = plt.subplots(3, 1, figsize=(12, 12))
ax = axes[0]
ax.plot(past.index, past.values, "o", color="#367a9c", ms=6, label="Recent observed rainfall")
forecast_mean = ensemble.mean(axis=1)
forecast_p10 = np.quantile(ensemble, .1, axis=1)
forecast_p90 = np.quantile(ensemble, .9, axis=1)
# Draw the percentile interval independently: the mean can lie outside it
# for a strongly skewed ensemble, so symmetric error bars are inappropriate.
ax.vlines(forecast.date, forecast_p10, forecast_p90, color="#d36c17", lw=1.5,
          label="Forecast ensemble 10–90% range")
ax.plot(forecast.date, forecast_p10, "_", color="#d36c17", ms=9)
ax.plot(forecast.date, forecast_p90, "_", color="#d36c17", ms=9)
ax.plot(forecast.date, forecast_mean, "o", color="#d36c17", ms=6, label="Forecast ensemble mean")
ax.axvline(issue, color="black", ls="--", alpha=0.35, label="Forecast date")
ax.set_ylabel("Daily rainfall (mm)"); ax.legend(fontsize=9); ax.grid(axis="y", alpha=.2)
# Accumulate each forecast member, then calculate the ensemble range.
# Date labels denote daily intervals; each point includes that day's rain.
observed_cumulative = past.cumsum()
forecast_cumulative = past.sum() + np.cumsum(ensemble, axis=0)
# Anchor the forecast curve to the last observed cumulative total.
cumulative_dates = pd.DatetimeIndex([past.index[-1]]).append(pd.DatetimeIndex(forecast.date))
cumulative_members = np.vstack([np.repeat(past.sum(), len(member_columns)), forecast_cumulative])
ax = axes[1]
ax.plot(past.index, observed_cumulative.values, "o-", color="#367a9c", label="Recent observed cumulative rainfall")
ax.plot(cumulative_dates, cumulative_members.mean(axis=1), "o-", color="#d36c17", label="Observed + forecast ensemble mean")
ax.fill_between(cumulative_dates, np.quantile(cumulative_members, .1, axis=1),
                np.quantile(cumulative_members, .9, axis=1), color="#d36c17", alpha=.2,
                label="Observed + forecast 10–90% range")
ax.axvline(issue, color="black", ls="--", alpha=0.35, label="Forecast date")
ax.axhline(RAINFALL_THRESHOLD_MM, color="#555555", ls=":", label="30 mm threshold")
ax.set_ylabel("Cumulative rainfall (mm)")
ax.set_xlabel("Date (daily interval; cumulative total includes this day)")
ax.set_ylim(bottom=0); ax.legend(fontsize=9); ax.grid(axis="y", alpha=.2)
ax = axes[2]
positions = np.arange(len(totals))
ax.bar(positions, np.repeat(past.sum(), len(totals)), color="#367a9c", label="Observed contribution")
ax.bar(positions, totals-past.sum(), bottom=past.sum(), color="#d36c17", label="Forecast contribution")
ax.axhline(RAINFALL_THRESHOLD_MM, color="black", ls="--", label="30 mm threshold")
ax.set_xticks(positions, [c.split("_")[1] for c in member_columns])
ax.set_xlabel("Forecast member"); ax.set_ylabel(f"{PAST_DAYS+FUTURE_DAYS}-day rainfall (mm)")
ax.legend(fontsize=9)
fig.suptitle(f"{COUNTRY} | {site.site_name} | forecast date {issue.date()}")
plt.tight_layout(); plt.show()
print("Forecast dates you can choose this season:", ", ".join(season_choices))

#### B. Result: does this window pass the SPW v2 rain test?

In [ ]:
v2_probability = prob_exceed_normal(totals.mean(), float(np.std(totals)), RAINFALL_THRESHOLD_MM)
v2_median = float(np.percentile(totals, 50))
verdict, colour = (("Rain test passed", "#367a9c") if v2_probability >= V2_P_SHORT else
                   ("Plant with caution", "#e0a800") if v2_median >= RAINFALL_THRESHOLD_MM else ("Wait", "#d98d64"))
phase = spw_phase(issue, COUNTRY)
phase_text = {"IN": "inside the planting window", "BEFORE": "before the planting window, when SPW only says <i>prepare</i>",
              "TOO_EARLY": "well before the planting window, when SPW gives no advice",
              "AFTER": "after the planting window, when a separate late-planting check applies"}[phase]
phase_lines = [f"This day is <b>{phase_text}</b>."]
advice_start = pd.Timestamp(f"{issue.year}-{SPW_V2[COUNTRY]['advice_floor']}")
if phase in ("BEFORE", "TOO_EARLY"):
    phase_lines.append("So SPW v2 would not advise planting on this day, whatever the rain test says. "
                       "Try a date inside the window (see Section 3).")
elif phase == "AFTER":
    phase_lines.append("After the window SPW does not use this rain test: it runs a short late-planting check instead "
                       "(1 observed + 3 forecast days, 5 mm, 50 %). Section 3 shows its answers.")
elif issue < advice_start:
    phase_lines.append(f"SPW is already checking (the watch date), but advice to farmers only starts on {advice_start:%d %b}.")
key_result(f"SPW v2 rain check on {issue:%d %b %Y}: {verdict}" + ("" if phase == "IN" else " (outside the planting window)"), [
    f"Rain already fallen (last {PAST_DAYS} days): <b>{past.sum():.1f} mm</b>. Rain the forecast expects "
    f"(next {FUTURE_DAYS} days, average of members): <b>{ensemble.sum(axis=0).mean():.1f} mm</b>.",
    f"Chance the total is above {RAINFALL_THRESHOLD_MM} mm: <b>{v2_probability:.0%}</b> (rain test passed needs "
    f"{V2_P_SHORT:.0%}). Middle (median) total: <b>{v2_median:.1f} mm</b> (plant with caution needs "
    f"{RAINFALL_THRESHOLD_MM} mm or more).",
    f"With hindsight, the rain that really fell over the window: <b>{past.sum() + observed_future.sum():.1f} mm</b>.",
] + phase_lines, colour=colour)
shown = pd.DataFrame({"Measure": ["Rain already fallen (mm)", "Rain the forecast expects, average (mm)",
                                  "Total, average of members (mm)", "Total, median of members (mm) — v2 test",
                                  f"Share of members reaching {RAINFALL_THRESHOLD_MM} mm",
                                  f"Chance of more than {RAINFALL_THRESHOLD_MM} mm (Normal curve) — v2 test",
                                  "Rain test result",
                                  "Rain that really fell over the window (mm; hindsight)"],
                      "Value": [f"{past.sum():.1f}", f"{ensemble.sum(axis=0).mean():.1f}", f"{totals.mean():.1f}",
                                f"{v2_median:.1f}", f"{(totals >= RAINFALL_THRESHOLD_MM).mean():.0%}",
                                f"{v2_probability:.0%}", verdict, f"{past.sum() + observed_future.sum():.1f}"]})
display(results_table(shown, "Details", reference_row=6))
print("This is one rain check on one date. Section 3 runs it for every day of the season, inside the planting window.")

> 💬 **Questions to discuss — Task 2**
>
> 1. How much rain had already fallen, and how much did the forecast expect? Which part mattered more for the answer?
> 2. How wide is the spread of the members (bottom chart)? What does a wide spread say about how sure the forecast is?
> 3. Change ISSUE_DATE to a date one or two weeks later and run the cells again. Does the answer change, and why?
> 4. With hindsight, was the forecast too wet, too dry or about right?
>
> ✍️ **Key question 1** (write your answer in Section 8): **Plant or wait on one day?** On the example date, did SPW say plant or wait? Was that decided more by the rain already fallen or by the 5-day forecast? *Quote:* the chance of more than 30 mm.

### 3. Through the season: when did SPW v2 first say plant?

> 🎯 **Objective:** follow SPW v2 every day of the season and find the first day it said plant.
>
> **You will learn to:**
> - read a season of SPW v2 answers: prepare, wait, plant with caution, rain test passed
> - find the first planting day and compare it with the original 40 mm rule
> - compare two seasons at the same place
>
> **Task sheet:** Task 3 · key question 2

**In short:** The rain check from Section 2, run for **every day** with the latest forecast available that day. SPW only advises inside the **planting window** from the season calendar; before it, it says *prepare*. The top row shows the **original rule** (40 mm) for comparison.

<details><summary><b>More detail</b> (click to open)</summary>

**Planting window** (GAP maize calendar): Chipata district (Zambia) from the 14 November *watch* date (advice to farmers
from 19 November; SPW starts checking on the watch date so that a signal is ready when advice opens) to 21 January;
Lilongwe district (Malawi) 21 November to 7 January; Nigeria's Guinea Savanna zone, including Kaduna, 1 May to 9 July.

**After the window** (Nigeria, July) GAP uses a separate late-planting check: 1 observed + 3 forecast days, chance of more
than 5 mm at least 50 %. Zambia and Malawi data end on 31 December, inside the window.

**The original rule** combines 13 overlapping 8- and 11-day windows, uses the forecast average only, and needs at least
6 of the 13 tests to pass plus a check against normal rainfall.

**Not included here** (needs data this notebook does not have): GAP's soil-moisture check (SPW v2.5) and the
long-term rainfall checks used before the window opens. Soil moisture is looked at afterwards in Section 5.

</details>

#### A. What SPW v2 said each day

Top: the original 40 mm rule (blue = plant) and the SPW v2 answer for each day, by colour. Bottom: the 11-day rain total SPW v2 checks (dots = median of the members, lines = 10–90 % range) against 30 mm; green shading = planting window.

In [ ]:
season_days = season_v2(COUNTRY, YEAR)
fig, axes = plt.subplots(2, 1, figsize=(12, 7.5), sharex=True, gridspec_kw={"height_ratios": [1, 1.6]})
ax = axes[0]
ax.scatter(season_days.index, np.repeat(1, len(season_days)), color=np.where(season_days.original_40, "#367a9c", "#eeeeee"), s=28)
for state, (label, colour) in STATE_STYLE.items():
    days = season_days.index[season_days.state.eq(state)]
    if len(days):
        ax.scatter(days, np.repeat(0, len(days)), color=colour, s=28, label=label,
                   edgecolors="#999999" if state == "NO_DATA" else "none")
ax.set_yticks([0, 1], ["SPW v2", "Original rule (40 mm)"])
ax.set_ylim(-.6, 1.6)
ax.legend(fontsize=8, ncol=4, loc="upper left", bbox_to_anchor=(0, -.12))
ax.set_title(f"{COUNTRY} | {site.site_name} | {YEAR} | planting window from the GAP calendar: {SPW_V2[COUNTRY]['calendar']}")
ax = axes[1]
in_window = season_days.phase.eq("IN")
ax.vlines(season_days.index, season_days.p10_mm, season_days.p90_mm, color="#367a9c", alpha=.35, lw=1, label="Members 10–90 % range")
ax.plot(season_days.index, season_days.median_mm, "o", ms=3.5, color="#367a9c", label="Median of the members")
ax.plot(season_days.index, season_days.observed_mm, "-", lw=1, color="#555555", label="Rain already fallen (6 days)")
ax.axhline(V2_THRESHOLD_MM, color="black", ls="--", lw=1, label="30 mm threshold")
if in_window.any():
    ax.axvspan(season_days.index[in_window].min(), season_days.index[in_window].max(), color="#d9e7cf", alpha=.45, zorder=0,
               label="Planting window")
ax.set_ylabel("11-day rain total (mm)")
ax.set_ylim(bottom=0); ax.legend(fontsize=8, loc="upper left"); ax.grid(axis="y", alpha=.2)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

#### B. Result: the first day SPW v2 said plant

In [ ]:
window_opens, window_closes = season_window(COUNTRY, YEAR)
first_ok = first_day(season_days.state.eq("RAIN_OK"))
first_caution = first_day(season_days.state.eq("PLANT_CAUTION"))
first_original = first_day(season_days.original_40)
lines = [f"SPW v2 first said <b>rain test passed</b> on <b>{show_day(first_ok)}</b>, and <b>plant with caution</b> on "
         f"<b>{show_day(first_caution)}</b>.",
         f"The original rule (40 mm) first said plant on <b>{show_day(first_original)}</b>."]
if pd.notna(first_ok) and pd.notna(first_original):
    gap = (first_original - first_ok).days
    lines.append(f"SPW v2 said plant <b>{abs(gap)} days {'earlier' if gap > 0 else 'later'}</b> than the original rule."
                 if gap else "Both said plant on the same day.")
advice_start = pd.Timestamp(f"{YEAR}-{SPW_V2[COUNTRY]['advice_floor']}")
lines.append(f"SPW checks from <b>{window_opens:%d %b}</b>"
             + (f" (the district's watch date; advice to farmers from {advice_start:%d %b})" if window_opens < advice_start else "")
             + f" until the planting window closes on <b>{window_closes:%d %b %Y}</b>."
             + (f" The daily SPW series in this practical ends on {season_days.index.max():%d %b %Y}, before the window closes."
                if season_days.index.max() < window_closes else ""))
if pd.notna(first_ok) and first_ok == window_opens:
    lines.append("The rain test passed on the <b>first day SPW checks</b>: the rain had already come before that, "
                 "when SPW only says <i>prepare</i>."
                 + (f" Farmers would first hear it on {advice_start:%d %b}, when advice starts." if window_opens < advice_start else ""))
key_result(f"{COUNTRY} {YEAR}: when SPW v2 first said plant", lines)
shown = pd.DataFrame({"SPW v2 answer": [STATE_STYLE[s][0] for s in STATE_STYLE if season_days.state.eq(s).any()],
                      "Days": [int(season_days.state.eq(s).sum()) for s in STATE_STYLE if season_days.state.eq(s).any()],
                      "First day": [show_day(first_day(season_days.state.eq(s))) for s in STATE_STYLE if season_days.state.eq(s).any()]})
display(results_table(shown, f"SPW v2 answers, {season_days.index.min():%d %b} to {season_days.index.max():%d %b %Y}",
                      label_colours={STATE_STYLE[s][0]: STATE_STYLE[s][1] for s in STATE_STYLE}, bar_column="Days"))

> 💬 **Questions to discuss — Task 3**
>
> 1. On which day did SPW v2 first say *rain test passed*, and *plant with caution*? How many days earlier or later than the original rule?
> 2. After the first *rain test passed*, did SPW keep saying plant, or did it go back to *wait*? What would you tell farmers on a day it changes?
> 3. Change YEAR at the top to the other season and use Run All. How different is the first planting day?
> 4. Why does SPW v2 say plant earlier than the original rule? (Think: one window or thirteen, 30 or 40 mm, how sure the forecast is.)
>
> ✍️ **Key question 2** (write your answer in Section 8): **When did SPW first say plant?** When did the rain test first pass, did SPW stick with it, and was it earlier than the original 40 mm rule? *Quote:* the first date in 2023 and in 2024.

### 4. How good was the forecast part of the rain total?

> 🎯 **Objective:** check SPW's 11-day rain total against the rain that really fell.
>
> **You will learn to:**
> - compare forecast totals with what fell, date by date
> - judge whether the forecast's range was too narrow or too wide
> - see how the length of the forecast part changes the match
>
> **Task sheet:** Task 4 · key question 3

**In short:** For every forecast date in the season, SPW's total (6 days already fallen + 5 forecast days) is plotted against the rain that really fell over the same 11 days. **Dots on the dashed line are a perfect match.** If the forecast judged its own uncertainty well, the rain that fell would be inside its 10–90 % range about 80 % of the time.

<details><summary><b>More detail</b> (click to open)</summary>

Both axes include the same rain already fallen, so the dots can look closer to the line than the forecast part alone
would. The average error, the typical miss and the range check are not affected: they measure the forecast days only. Neighbouring dates share days, so they are not independent.

</details>

In [ ]:
# ✏️ Settings: how many forecast days to add (SPW v2 uses 5; try 3 and 10)
VERIFY_FUTURE_DAYS = 5   # 1–10

#### A. SPW's rain total against the rain that really fell

Each dot is one forecast date. The vertical line through a dot is the members' 10–90 % range.

In [ ]:
if not 1 <= VERIFY_FUTURE_DAYS <= 10:
    raise ValueError("Use 1 to 10 forecast days.")
VERIFY_PAST_DAYS = V2_PAST_DAYS
verification = splice_check(COUNTRY, YEAR, VERIFY_PAST_DAYS, VERIFY_FUTURE_DAYS)
fig, ax = plt.subplots(figsize=(7, 6))
ax.vlines(verification.observed_total_mm, verification.spliced_p10_mm, verification.spliced_p90_mm,
          color="#367a9c", alpha=.5, label="Forecast members 10–90 % range")
ax.scatter(verification.observed_total_mm, verification.spliced_ensemble_mean_mm,
           color="#367a9c", label="SPW total (average of members)")
limit = max(1., verification.observed_total_mm.max(), verification.spliced_p90_mm.max(),
            verification.spliced_ensemble_mean_mm.max())*1.05
ax.plot([0,limit], [0,limit], "--", color="grey", label="Perfect match")
ax.set(xlim=(0,limit), ylim=(0,limit), xlabel=f"Rain that really fell, {VERIFY_PAST_DAYS+VERIFY_FUTURE_DAYS} days (mm)",
       ylabel=f"SPW total: {VERIFY_PAST_DAYS} days fallen + {VERIFY_FUTURE_DAYS} forecast days (mm)",
       title=f"{COUNTRY} | {YEAR} season | {len(verification)} forecast dates")
ax.set_aspect("equal"); ax.legend(fontsize=8); ax.grid(alpha=.15)
plt.tight_layout(); plt.show()

#### B. Result: how well did the rain total match?

In [ ]:
errors = verification.spliced_ensemble_mean_mm - verification.observed_total_mm
inside = verification.observed_total_mm.between(verification.spliced_p10_mm, verification.spliced_p90_mm).mean()
key_result(f"{COUNTRY} {YEAR}: {len(verification)} forecast dates, {VERIFY_FUTURE_DAYS} forecast days", [
    f"On average SPW's total was <b>{abs(errors.mean()):.1f} mm {'lower' if errors.mean() < 0 else 'higher'}</b> "
    f"than the rain that fell. A typical miss, either way, was <b>{errors.abs().mean():.1f} mm</b>.",
    f"The rain that fell was inside the forecast's 10–90 % range on <b>{inside:.0%}</b> of dates "
    "(a well-judged range would hold it about 80 % of the time).",
    "Both totals include the same rain already fallen, so the scatter can look tighter than the forecast alone; the average "
    "error, typical miss and range check above measure only the forecast days.",
])
shown = pd.DataFrame({"Forecast date": verification.initialization_date.dt.strftime("%d %b %Y"),
                      "Window": verification.window_start.dt.strftime("%d %b") + " – " + verification.window_end.dt.strftime("%d %b"),
                      "Rain already fallen (mm)": verification.recent_observed_mm,
                      "SPW total: average (mm)": verification.spliced_ensemble_mean_mm,
                      "SPW total: 10–90 % range (mm)": verification.spliced_p10_mm.map("{:.0f}".format) + " – "
                                                       + verification.spliced_p90_mm.map("{:.0f}".format),
                      "Rain that really fell (mm)": verification.observed_total_mm,
                      "Miss (mm)": errors})
styled = (results_table(shown, "Every forecast date (scroll for more)")
          .format("{:.1f}", subset=["Rain already fallen (mm)", "SPW total: average (mm)", "Rain that really fell (mm)"])
          .format("{:+.1f}", subset=["Miss (mm)"])
          .background_gradient(subset=["Miss (mm)"], cmap="RdBu", vmin=-40, vmax=40))
display(HTML(f"<div style='max-height:360px;overflow-y:auto'>{styled.to_html()}</div>"))

> 💬 **Questions to discuss — Task 4**
>
> 1. Was SPW's total usually too wet or too dry? By how much on a typical day?
> 2. On how many dates was the rain that fell inside the 10–90 % range, compared with the 80 % a well-judged range would give? If not, what could explain it: a range that is too narrow, a forecast that is too wet or too dry, or both?
> 3. Set VERIFY_FUTURE_DAYS to 3 and then 10. Does the match get better or worse, and why?
> 4. Which costs a farmer more: SPW saying plant when the rain does not come (a false start), or saying wait when it does?
>
> ✍️ **Key question 3** (write your answer in Section 8): **Can we trust the forecast part?** How much can SPW rely on the 5-day forecast? *Quote:* how often the real rain fell inside the forecast range (about 8 times in 10 is good).

### 5. What was the soil like after SPW said plant?

> 🎯 **Objective:** check what happened to the soil after the days SPW v2 said plant, and after the days it said wait.
>
> **You will learn to:**
> - compare soil moisture in the 15 days after *rain test passed*, *plant with caution* and *wait* days
> - compare it with the soil-moisture threshold GAP uses for your country
> - see why GAP's current SPW (v2.5) adds a soil check on the day
>
> **Task sheet:** Task 5 · key question 4

**In short:** Soil moisture is given as **% of field capacity** (the water a soil holds once excess water has drained); low values mean dry soil, risky for seedlings. Each day has one soil value summarising the **next 15 days** (supplied with the original toolkit's data). For each day, the chart shows that value, coloured by what SPW v2 said that day. The green line is GAP's soil threshold for your country (45 % in Zambia and Malawi, 70 % in Nigeria).

<details><summary><b>More detail</b> (click to open)</summary>

The soil values describe the 15 days after each day, so they check the outcome; SPW v2 never sees them. The toolkit's
data give one summary value per day as % of field capacity; the soil model and depth are those of the original toolkit.
GAP's current SPW (v2.5) checks the soil **on** the day instead, which these data do not include, so this section cannot
say which days a v2.5 soil check would have stopped. Later days in the season are usually wetter anyway, and neighbouring days share most of
their 15 days, so treat differences as a pattern to discuss, not a proof.

</details>

#### A. Soil moisture after each day

Top: daily rain. Middle: soil moisture over the next 15 days, coloured by the SPW v2 answer that day. Bottom: the same values grouped by answer, for days inside the planting window (black line = median).

In [ ]:
soil_days = season_v2(COUNTRY, YEAR)
soil_gate = SOIL_THRESHOLD[COUNTRY]
rain = obs.set_index("date").rainfall_mm.reindex(soil_days.index)
fig, axes = plt.subplots(3, 1, figsize=(12, 12), gridspec_kw={"height_ratios": [1, 1.3, 1.4]})
axes[0].bar(soil_days.index, rain, color="#889ca5")
axes[0].set_ylabel("Daily rainfall (mm)"); axes[0].set_title(f"{COUNTRY} | {site.site_name} | {YEAR}: observed rain")
axes[1].plot(soil_days.index, soil_days.soil_next_15_days, color="#aaaaaa", lw=1, zorder=1)
for state, (label, colour) in STATE_STYLE.items():
    mask = soil_days.state.eq(state) & soil_days.soil_next_15_days.notna()
    if mask.any():
        axes[1].scatter(soil_days.index[mask], soil_days.soil_next_15_days[mask], color=colour, s=24, label=label, zorder=2,
                        edgecolors="#999999" if state in ("NO_DATA", "TOO_EARLY") else "none")
axes[1].set_ylabel("Soil moisture, next 15 days\n(% of field capacity)"); axes[1].set_ylim(0, 105)
axes[1].legend(fontsize=8, ncol=3, loc="upper left")
for ax in axes[:2]:
    ax.set_xlim(soil_days.index.min() - pd.Timedelta(days=1), soil_days.index.max() + pd.Timedelta(days=1))
    ax.grid(axis="y", alpha=.2); ax.set_axisbelow(True)
SOIL_GROUPS = [("RAIN_OK", "Rain test passed"), ("PLANT_CAUTION", "Plant with caution"), ("WAIT", "Wait")]
inside = soil_days.phase.eq("IN")
soil_rows = []
for position, (state, label) in enumerate(SOIL_GROUPS, 1):
    values = soil_days.soil_next_15_days[inside & soil_days.state.eq(state)].dropna()
    if len(values):
        jitter = np.random.default_rng(1).uniform(-.15, .15, len(values))
        axes[2].scatter(position + jitter, values, color=STATE_STYLE[state][1], s=40, edgecolor="white", lw=.6, zorder=4)
        axes[2].hlines(values.median(), position - .28, position + .28, color="black", lw=2, zorder=3)
        axes[2].text(position + .32, values.median(), f"median {values.median():.0f} %", va="center", fontsize=10,
                     fontweight="bold")
    else:
        axes[2].text(position, 50, "no days", ha="center", color="#777777")
    soil_rows.append({"SPW v2 said": label, "Days with soil data": len(values),
                      "Median soil moisture, next 15 days (%)": values.median() if len(values) else np.nan,
                      f"Days below {soil_gate} %": int(values.lt(soil_gate).sum())})
soil_table = pd.DataFrame(soil_rows)
axes[2].set_xticks(range(1, 4), [f"{r['SPW v2 said']}\nn={r['Days with soil data']} day{'s' if r['Days with soil data'] != 1 else ''}" for r in soil_rows])
axes[2].set_xlim(.4, 3.9); axes[2].set_ylim(0, 105)
axes[2].set_ylabel("Soil moisture, next 15 days\n(% of field capacity)")
axes[2].set_title("Days inside the planting window, grouped by the SPW v2 answer")
for ax in axes[1:]:
    ax.axhline(soil_gate, color="#2e8b57", ls="--", lw=1.5, zorder=1)
    ax.text(1.0, soil_gate + 2, f"GAP soil threshold, {COUNTRY}: {soil_gate} %", transform=ax.get_yaxis_transform(),
            ha="right", fontsize=9, color="#2e8b57")
    ax.grid(axis="y", alpha=.2)
plt.tight_layout(); plt.show()

#### B. Result: soil moisture after *rain test passed* and after *wait* days

In [ ]:
rows = soil_table.set_index("SPW v2 said")
lines = []
for label in ["Rain test passed", "Plant with caution", "Wait"]:
    r = rows.loc[label]
    if r["Days with soil data"]:
        lines.append(f"After <b>{label.lower()}</b> days ({int(r['Days with soil data'])} with soil data): median soil moisture "
                     f"<b>{r['Median soil moisture, next 15 days (%)']:.0f} %</b> of field capacity; "
                     f"{int(r[f'Days below {soil_gate} %'])} of them below GAP's {soil_gate} % threshold.")
    else:
        lines.append(f"No <b>{label.lower()}</b> days inside the planting window this season.")
if 0 < rows.loc["Rain test passed", "Days with soil data"] < 10:
    lines.append("Only a few <i>rain test passed</i> days: read this comparison with care.")
lines.append("This is an association: it does not show that SPW caused the difference, and later days in the season are usually wetter anyway.")
key_result(f"{COUNTRY} {YEAR}: soil moisture after each SPW v2 answer (inside the planting window)", lines)
display(results_table(soil_table, "Soil moisture over the 15 days after each day",
                      label_colours={STATE_STYLE[s][0]: STATE_STYLE[s][1] for s, _ in SOIL_GROUPS},
                      bar_column="Median soil moisture, next 15 days (%)")
        .format({"Median soil moisture, next 15 days (%)": "{:.0f}"}, na_rep="—"))

> 💬 **Questions to discuss — Task 5**
>
> 1. Was the soil wetter after *rain test passed* days than after *wait* days? By how much?
> 2. Were there *rain test passed* days followed by soil below GAP's threshold? What could have happened to a farmer who planted on those days?
> 3. Compare the two seasons (change YEAR at the top). Does the pattern hold?
> 4. GAP's current SPW (v2.5) also checks the soil on the day itself, which these data do not have. Why might a soil check on the day change the advice?
>
> ✍️ **Key question 4** (write your answer in Section 8): **Was the soil ready?** After the days SPW said plant, was the soil usually wet enough for the crop? *Quote:* the median soil moisture after those days, against GAP's threshold (45 %, or 70 % in Nigeria).

### 6. Malawi farms: did planting in line with SPW go with better yields? (everyone)

> 🎯 **Objective:** compare the yields of real farms that planted before, in line with, or after SPW.
>
> **You will learn to:**
> - compare farm yields by planting date
> - read a 90 % likely range and say when data cannot separate two groups
> - check whether a result holds when the settings change
>
> **Task sheet:** Task 6 · key question 5 (every team, whatever its country)

**In short:** This section always uses **Malawi 2023/24**: One Acre Fund farms near the Lilongwe point (open data, GROW-Africa). Farms that planted within a few days of the first SPW v2 *rain test passed* day (27 November 2023) planted **in line with SPW**; the others planted **before** or **after**. We also compare farms planting around the SPW v2 date with farms planting around the original rule's later date (18 December).

<details><summary><b>More detail</b> (click to open)</summary>

**Farm data.** One Acre Fund's revised planting-date survey records, 2023/24 season, farms within 100 km of the
Lilongwe point. Each record has the farmer-reported planting date and yield (t/ha); no names or coordinates. The nearest
farm is 32 km away. The records are treated as maize because their yields match One Acre Fund's maize survey.

**Method.** Mean yields are compared. The **90 % likely range** comes from resampling the farms 2,000 times: if it
includes 0 %, this data cannot clearly tell higher yields from lower ones.

**How strong is this evidence?** These farmers were not sent SPW and chose their own dates; farms that plant early or
late may also differ in soil, inputs or labour. It is one season, with one rainfall grid cell for every farm, and
dates and yields are farmer-reported. So this can show whether the pattern is *consistent* with SPW, not prove that
following SPW raises yields.

</details>

In [ ]:
# ✏️ Settings (always Malawi 2023/24)
RADIUS_KM = 50       # farms within this distance of the Lilongwe point (35–100 km)
BASELINE_DAYS = 5    # "in line with SPW" = planted within this many days of the SPW date (1–15)

In [ ]:
# Prepare the Malawi farms, the SPW dates and the two result tables (shown in C and D)
COMPARE_TO = "v2 rain test passed"
if COUNTRY != "Malawi":
    print(f"Note: this use case uses Malawi farm data, so it shows Malawi 2023/24 (not {COUNTRY}).")
use_country("Malawi")
support = pd.read_csv(DATA_DIR / "malawi_forecast_support.csv", parse_dates=["initialization_date", "date"])
farms = pd.read_csv(DATA_DIR / "malawi_farms_2023_24.csv", parse_dates=["planting_date"])
if not 35 <= RADIUS_KM <= 100 or not 1 <= BASELINE_DAYS <= 15:
    raise ValueError("Use a radius of 35–100 km (the nearest farm is 32 km away) and 1–15 baseline days.")
farms = farms[farms.distance_km.le(RADIUS_KM) & farms.planting_date.notna() & farms.yield_t_ha.notna()].copy()
if farms.empty:
    raise ValueError(f"No farms within {RADIUS_KM} km. Use a larger radius.")

# SPW dates for the 2023/24 season at the Lilongwe point
season = spw[spw.date.between("2023-09-01", "2023-12-31")].reset_index(drop=True)
malawi_days = pd.DataFrame([spw_v2_day(r.date, r.forecast_initialization_date, "Malawi", support)
                   for _, r in season.iterrows()])
spw_dates = {"v2 rain test passed": malawi_days.date[malawi_days.state.eq("RAIN_OK")].min(),
             "v2 plant with caution": malawi_days.date[malawi_days.state.eq("PLANT_CAUTION")].min(),
             "Original 40 mm": season.date[season.spw_40.eq(1)].min()}
if COMPARE_TO not in spw_dates:
    raise ValueError(f"COMPARE_TO must be one of: {', '.join(spw_dates)}")
reference = spw_dates[COMPARE_TO]
if pd.isna(reference):
    raise ValueError(f"'{COMPARE_TO}' did not occur in this season. Choose another COMPARE_TO.")

GROUPS = ["Before SPW", "In line with SPW", "After SPW"]
group_colours = {"Before SPW": "#d98d64", "In line with SPW": "#367a9c", "After SPW": "#9e9e9e"}
lag = (farms.planting_date - reference).dt.days
farms["group"] = np.select([lag < -BASELINE_DAYS, lag > BASELINE_DAYS], ["Before SPW", "After SPW"], "In line with SPW")
rng = np.random.default_rng(0)

def likely_range(in_a, in_b, n=2000):
    """90 % range of the % difference in mean yield, group a vs group b.
    Farms are resampled together, so a farm that belongs to both groups stays in both."""
    in_a, in_b, y = in_a.to_numpy(), in_b.to_numpy(), farms.yield_t_ha.to_numpy()
    if in_a.sum() < 5 or in_b.sum() < 5:
        return np.nan, np.nan
    diffs = []
    for _ in range(n):
        pick = rng.integers(0, len(y), len(y))
        a, b = in_a[pick], in_b[pick]
        if a.any() and b.any():
            diffs.append(y[pick][a].mean() / y[pick][b].mean() - 1)
    return tuple(np.percentile(diffs, [5, 95]) * 100)

def comparison_row(label, in_group, in_base, is_base):
    """One result row: farms, mean yield, % difference from the reference group and its 90 % likely range."""
    y, base = farms.yield_t_ha[in_group], farms.yield_t_ha[in_base]
    row = {**label, "Farms": len(y), "Mean yield (t/ha)": y.mean() if len(y) else np.nan,
           "Difference (%)": np.nan, "Range low (%)": np.nan, "Range high (%)": np.nan, "Reference": is_base}
    if not is_base and len(y) and len(base):
        row["Difference (%)"] = (y.mean() / base.mean() - 1) * 100
        row["Range low (%)"], row["Range high (%)"] = likely_range(in_group, in_base)
    return row

# Result 1: before / in line with / after the chosen SPW date
in_line = farms.group.eq("In line with SPW")
before_after_table = pd.DataFrame([comparison_row({"Planted": g, "Share of farms (%)": 100 * farms.group.eq(g).mean()},
                                                  farms.group.eq(g), in_line, g == "In line with SPW")
                                   for g in GROUPS])
# Result 2: farms planting around each SPW signal, compared with the original 40 mm date
near = {name: (farms.planting_date - day).dt.days.abs().le(BASELINE_DAYS)
        for name, day in spw_dates.items() if pd.notna(day)}
in_original = near.get("Original 40 mm", pd.Series(False, index=farms.index))
signal_table = pd.DataFrame([comparison_row({"SPW signal": name, "Date": f"{spw_dates[name]:%d %b %Y}"},
                                            mask, in_original, name == "Original 40 mm")
                             for name, mask in near.items()])

key_result("Malawi 2023/24: what is being compared", [
    f"<b>{len(farms)} farms</b> within {RADIUS_KM} km of {site.site_name}.",
    f"<b>In line with SPW</b> = planted within ±{BASELINE_DAYS} days of <b>{COMPARE_TO}</b> ({reference:%d %b %Y}).",
    "SPW dates this season: " + "; ".join(f"{name} <b>{day:%d %b}</b>" for name, day in spw_dates.items() if pd.notna(day)) + ".",
])

#### A. When did farms plant, compared with the SPW dates?

Each bar is the number of farms that planted that day, coloured by group. The shaded band is *in line with SPW*. The green line is the observed rain; the black lines mark the SPW dates.

In [ ]:
line_styles = {"v2 rain test passed": "-", "v2 plant with caution": "--", "Original 40 mm": ":"}
days = pd.date_range("2023-10-01", "2024-01-15")
daily = farms.groupby(["planting_date", "group"]).size().unstack(fill_value=0).reindex(days, fill_value=0)
fig, ax = plt.subplots(figsize=(12, 4.8))
bottom = np.zeros(len(days))
for group, colour in group_colours.items():
    if group in daily:
        ax.bar(days, daily[group], bottom=bottom, width=1, color=colour, label=f"Planted {group[0].lower()}{group[1:]}")
        bottom += daily[group].to_numpy()
ax.axvspan(reference - pd.Timedelta(days=BASELINE_DAYS), reference + pd.Timedelta(days=BASELINE_DAYS),
           color="#367a9c", alpha=.12, zorder=0, label=f"In line with SPW (±{BASELINE_DAYS} days)")
for name, day in spw_dates.items():
    if pd.notna(day):
        ax.axvline(day, color="black", ls=line_styles[name], lw=1.2, label=f"{name}: {day:%d %b}")
ax.set_ylabel("Farms planting that day")
rain_ax = ax.twinx()
rain_ax.plot(days, obs.set_index("date").rainfall_mm.reindex(days), color="#2b6f2b", lw=1, alpha=.7,
             label="Observed rainfall (Lilongwe grid cell)")
rain_ax.set_ylabel("Rainfall (mm/day)")
handles, labels = [sum(pair, []) for pair in zip(ax.get_legend_handles_labels(), rain_ax.get_legend_handles_labels())]
ax.legend(handles, labels, fontsize=9, loc="upper left", bbox_to_anchor=(1.07, 1), frameon=False)
ax.set_title(f"When {len(farms)} farms within {RADIUS_KM} km of {site.site_name} planted, 2023/24", loc="left")
ax.grid(axis="y", alpha=.2); ax.set_axisbelow(True)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

#### B. Yield by planting week

Each dot is the mean yield of the farms that planted in that week. Bigger dots mean more farms; faded dots have fewer than 5 farms.

In [ ]:
weekly = (farms.groupby(pd.Grouper(key="planting_date", freq="7D", origin=pd.Timestamp("2023-10-02")))
          .yield_t_ha.agg(["mean", "size"]))
weekly = weekly[weekly["size"] > 0]
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.axvspan(reference - pd.Timedelta(days=BASELINE_DAYS), reference + pd.Timedelta(days=BASELINE_DAYS),
           color="#367a9c", alpha=.12, zorder=0, label=f"In line with SPW (±{BASELINE_DAYS} days)")
ax.scatter(weekly.index + pd.Timedelta(days=3), weekly["mean"], s=weekly["size"] * 6, color="#367a9c",
           alpha=np.where(weekly["size"] >= 5, .9, .3), label="Mean yield of farms planting that week")
for name, day in spw_dates.items():
    if pd.notna(day):
        ax.axvline(day, color="black", ls=line_styles[name], lw=1.2, label=f"{name}: {day:%d %b}")
ax.set_xlim(days[0], days[-1]); ax.set_ylabel("Yield (t/ha)")
ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, markerscale=.5)
ax.set_title("Mean yield by planting week (dot size = number of farms)", loc="left")
ax.grid(axis="y", alpha=.2); ax.set_axisbelow(True)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

#### C. Result 1: farms that planted before, in line with, or after SPW

The blue row is the reference: farms that planted in line with SPW. **Difference** is how much higher (green) or lower (red) each other group's mean yield was. If the **90 % likely range** includes 0, this data cannot clearly separate the groups.

In [ ]:
def pct_range(low, high):
    """'-22 to +2' style; an end that rounds to 0 but is not 0 keeps one decimal (-0.2), so 'includes 0' stays readable."""
    end = lambda x: f"{x:+.1f}" if round(x) == 0 and x != 0 else f"{round(x):+d}"
    return f"{end(low)} to {end(high)}"


def range_cells(table):
    """Text for the Difference, 90 % likely range and Clear difference? columns."""
    difference, likely, clear = [], [], []
    for _, r in table.iterrows():
        if r["Reference"]:
            difference.append("reference"); likely.append("—"); clear.append("—")
        elif pd.isna(r["Difference (%)"]):
            difference.append("—"); likely.append("—"); clear.append("—")
        elif pd.isna(r["Range low (%)"]):
            difference.append(f"{r['Difference (%)']:+.1f} %"); likely.append("too few farms"); clear.append("—")
        else:
            difference.append(f"{r['Difference (%)']:+.1f} %")
            likely.append(f"{pct_range(r['Range low (%)'], r['Range high (%)'])} %")
            clear.append("Yes" if r["Range low (%)"] > 0 or r["Range high (%)"] < 0 else "No — includes 0")
    return difference, likely, clear

def finding(name, r, base_name):
    """One plain-language sentence for the key-result box."""
    if pd.isna(r["Difference (%)"]):
        return f"No farms planted {name.lower()} in this setting."
    d = r["Difference (%)"]
    text = (f"<b>{name}</b> ({r['Farms']} farms): mean yield {r['Mean yield (t/ha)']:.2f} t/ha, "
            f"<b>{abs(d):.0f} % {'higher' if d > 0 else 'lower'}</b> than {base_name}.")
    if pd.isna(r["Range low (%)"]):
        return text + " Too few farms for a likely range."
    if r["Range low (%)"] > 0 or r["Range high (%)"] < 0:
        return text + f" The 90 % likely range ({pct_range(r['Range low (%)'], r['Range high (%)'])} %) excludes 0."
    return text + (f" But the 90 % likely range ({pct_range(r['Range low (%)'], r['Range high (%)'])} %) "
                   "includes 0, so this data cannot clearly separate the two.")

t = before_after_table
base = t[t.Reference].iloc[0]
key_result("Key result", [finding(f"Planted {r.Planted[0].lower()}{r.Planted[1:]}", r,
                                  f"farms in line with SPW ({base['Mean yield (t/ha)']:.2f} t/ha)")
                          for _, r in t[~t.Reference].iterrows()])
difference, likely, clear = range_cells(t)
shown = pd.DataFrame({"Planted": t.Planted, "Farms": t.Farms,
                      "Share of farms": t["Share of farms (%)"].map("{:.0f} %".format),
                      "Mean yield (t/ha)": t["Mean yield (t/ha)"].round(2),
                      "Difference": difference, "90 % likely range": likely, "Clear difference?": clear})
display(results_table(shown, f"Yield compared with farms that planted in line with SPW "
                             f"(±{BASELINE_DAYS} days of {COMPARE_TO}, {reference:%d %b %Y})",
                      difference=t["Difference (%)"], reference_row=int(t.index[t.Reference][0]),
                      label_colours=group_colours, bar_column="Mean yield (t/ha)")
        .format({"Mean yield (t/ha)": "{:.2f}"}, na_rep="—"))
if (t.Farms.between(1, 9)).any():
    print("A group has fewer than 10 farms: treat its result with caution.")

#### D. Result 2: the SPW v2 date against the original rule's date

Farms planting around each SPW date, compared with farms planting around the original 40 mm date (blue row). With a wide BASELINE_DAYS the windows overlap, so a farm can count for two dates.

In [ ]:
t = signal_table
if not t.Reference.any():
    print("The original 40 mm rule did not trigger this season, so there is no reference row.")
else:
    base = t[t.Reference].iloc[0]
    key_result("Key result", [finding(f"Planted around '{r['SPW signal']}' ({r.Date[:6]})", r,
                                      f"farms around the original 40 mm date ({base['Mean yield (t/ha)']:.2f} t/ha)")
                              for _, r in t[~t.Reference].iterrows()])
difference, likely, clear = range_cells(t)
shown = pd.DataFrame({"SPW signal": t["SPW signal"], "Date": t.Date, "Farms": t.Farms,
                      "Mean yield (t/ha)": t["Mean yield (t/ha)"].round(2),
                      "Difference": difference, "90 % likely range": likely, "Clear difference?": clear})
reference_row = int(t.index[t.Reference][0]) if t.Reference.any() else None
display(results_table(shown, f"Yield of farms planting within ±{BASELINE_DAYS} days of each SPW signal, "
                             "compared with the original 40 mm date",
                      difference=t["Difference (%)"], reference_row=reference_row, bar_column="Mean yield (t/ha)")
        .format({"Mean yield (t/ha)": "{:.2f}"}, na_rep="—"))
if (t.Farms.between(1, 9)).any():
    print("A signal has fewer than 10 farms: treat its result with caution.")
# ---- What this means: one summary drawn from both results -----------------------------------
before = before_after_table.set_index("Planted").loc["Before SPW", "Difference (%)"]
v2_vs_original = (signal_table.set_index("SPW signal")["Difference (%)"].get("v2 rain test passed", np.nan)
                  if signal_table.Reference.any() else np.nan)
def clarity(r):
    if pd.isna(r["Difference (%)"]):
        return None
    if pd.isna(r["Range low (%)"]):
        return "but too few farms for a likely range"
    if r["Range low (%)"] > 0 or r["Range high (%)"] < 0:
        return "a clear difference (the 90 % likely range excludes 0)"
    return "but the 90 % likely range includes 0, so the data cannot confirm it"


meaning = []
before_row = before_after_table.set_index("Planted").loc["Before SPW"]
if clarity(before_row):
    d = before_row["Difference (%)"]
    meaning.append(f"Farms that planted <b>before</b> SPW reported <b>{abs(d):.0f} % {'lower' if d < 0 else 'higher'}</b> yields "
                   f"than farms that planted in line with SPW: {clarity(before_row)}.")
v2_row = signal_table.set_index("SPW signal").loc["v2 rain test passed"] if signal_table.Reference.any() else None
if v2_row is not None and clarity(v2_row):
    d = v2_row["Difference (%)"]
    meaning.append(f"Farms planting around the <b>SPW v2 date</b> reported <b>{abs(d):.0f} % {'higher' if d > 0 else 'lower'}</b> "
                   f"yields than farms around the original rule's later date: {clarity(v2_row)}.")
points_the_spw_way = (pd.notna(before) and before < 0 and pd.notna(v2_vs_original) and v2_vs_original > 0)
meaning.append(("Both point the way SPW would predict. " if points_the_spw_way else
                "These numbers do not both point the way SPW would predict. ")
               + "Either way, one season cannot prove that following SPW raises yields: farmers chose their own dates, "
               "and early and late planters may differ in other ways.")
meaning.append("To confirm it: more seasons, more places, and farmers who actually receive SPW.")
key_result("What this means", meaning, colour="#2e8b57")

> 💬 **Questions to discuss — Task 6**
>
> 1. How many farms planted **before** SPW v2 first said the rain was enough? What did the rain do in the weeks after they planted (chart A)?
> 2. Did farms that planted before or after SPW report lower yields than farms in line with SPW? Does each 90 % likely range include 0?
> 3. Did farms planting around the SPW v2 date report higher or lower yields than farms around the original rule's date?
> 4. Change BASELINE_DAYS to 3, 7 and 10, and RADIUS_KM to 40, 75 and 100. Which results hold, and which depend on the settings?
> 5. What data would you need to run the same check in your own country?
>
> ✍️ **Key question 5** (write your answer in Section 8): **Did following SPW pay off?** Did Malawi farms that planted when SPW said plant get better yields than other farms, and how sure can we be? *Quote:* a yield difference and whether its range includes 0.

### 7. What we learned

The box below summarises the main numbers for your example point in **both seasons**, using the default settings.
Run it after choosing the country in Section 1.

In [ ]:
summary_rows = []
for year in (2023, 2024):
    s = season_v2(COUNTRY, year)
    ok, original = first_day(s.state.eq("RAIN_OK")), first_day(s.original_40)
    check = splice_check(COUNTRY, year)
    inside = check.observed_total_mm.between(check.spliced_p10_mm, check.spliced_p90_mm).mean()
    in_window = s.phase.eq("IN")
    soil_ok = s.soil_next_15_days[in_window & s.state.eq("RAIN_OK")].median()
    soil_wait = s.soil_next_15_days[in_window & s.state.eq("WAIT")].median()
    summary_rows.append({"Season": year, "SPW v2: first rain test passed": show_day(ok),
                         "Original rule (40 mm): first plant": show_day(original),
                         "SPW v2 earlier by (days)": (original - ok).days if pd.notna(ok) and pd.notna(original) else "—",
                         "Rain that fell inside the forecast range": f"{inside:.0%}",
                         "Soil after rain test passed / wait (median %)": f"{soil_ok:.0f} / {soil_wait:.0f}"
                         if pd.notna(soil_ok) and pd.notna(soil_wait) else "—"})
learn = pd.DataFrame(summary_rows)
display(results_table(learn.astype(str), f"{COUNTRY} ({site.site_name}): the main numbers from this practical (default settings)"))
key_result(f"What we learned for {COUNTRY}", [
    "SPW v2 first said <b>rain test passed</b> on " + " and ".join(f"<b>{r['SPW v2: first rain test passed']}</b>"
                                                                  for r in summary_rows) + ".",
    "The original 40 mm rule said plant " + " and ".join(
        (f"{r['SPW v2 earlier by (days)']} days later" if r["SPW v2 earlier by (days)"] != "—" and r["SPW v2 earlier by (days)"] >= 0
         else f"{-r['SPW v2 earlier by (days)']} days earlier" if r["SPW v2 earlier by (days)"] != "—" else "(no comparison)")
        + f" in {r['Season']}" for r in summary_rows) + ".",
    "The rain that fell landed inside the forecast's 10–90 % range on " + " and ".join(
        r["Rain that fell inside the forecast range"] for r in summary_rows) + " of dates, below the 80 % a well-judged range would give: the forecast "
    "misses the rain more often than it should (its range is too narrow, or it is too wet or too dry), so advice should "
    "keep a margin of caution.",
    f"GAP's soil threshold for {COUNTRY} is {SOIL_THRESHOLD[COUNTRY]} % of field capacity: compare it with the soil after "
    "<i>rain test passed</i> days in the table.",
    "Malawi farms (Section 6): see your answer to key question 5."], colour="#2e8b57")

#### Key learnings

| Section | Learning |
|---|---|
| 2 · One rain check | SPW v2 adds the rain already fallen to the forecast, and asks **how sure** the forecast is, not just how much rain it expects on average. |
| 3 · Through the season | SPW v2 usually says plant **earlier** than the original rule. Earlier is only better if the rain keeps coming, which is what Sections 4–6 check. |
| 4 · Forecast quality | The forecast part can be too wet or too dry, and real rain falls outside its range more often than it should (too narrow a range, a bias, or both). |
| 5 · Soil | Rain-based advice can be followed by soil that is still dry. This is why GAP's current SPW (v2.5) adds a soil check on the day. |
| 6 · Malawi farms | Farm records can show whether planting in line with SPW is **consistent** with better yields; proving it needs more seasons and farmers who receive SPW. |

> ✅ **The main message:** a planting signal built from recent rain plus a short-range forecast can be given *earlier*
> than the old rule, but it should say how sure it is: *"the rains have started and more is likely this week; if your
> soil is moist, you can plant"*.
>
> ⚠️ **Too strong a conclusion:** *"SPW says plant, so the crop is safe"*. These are single points, two seasons and
> farmer-reported yields; the soil and dry spells after planting still matter.

### 8. Your answers and planting window (group output)

> 🎯 **Objective:** bring together what you found and turn it into a planting window you could defend in front of the room.
>
> **Task sheet:** Group output · everyone

**Part A: the five key questions.** One short answer each, with at least one number or date from the notebook.

| | Key question | Section |
|---|---|---|
| Q1 | **Plant or wait on one day?** On the example date, did SPW say plant or wait? Was that decided more by the rain already fallen or by the 5-day forecast? *Quote:* the chance of more than 30 mm. | 2 |
| Q2 | **When did SPW first say plant?** When did the rain test first pass, did SPW stick with it, and was it earlier than the original 40 mm rule? *Quote:* the first date in 2023 and in 2024. | 3 |
| Q3 | **Can we trust the forecast part?** How much can SPW rely on the 5-day forecast? *Quote:* how often the real rain fell inside the forecast range (about 8 times in 10 is good). | 4 |
| Q4 | **Was the soil ready?** After the days SPW said plant, was the soil usually wet enough for the crop? *Quote:* the median soil moisture after those days, against GAP's threshold (45 %, or 70 % in Nigeria). | 5 |
| Q5 | **Did following SPW pay off?** Did Malawi farms that planted when SPW said plant get better yields than other farms, and how sure can we be? *Quote:* a yield difference and whether its range includes 0. | 6 |

**Part B: your planting window** for the example location, to present in **3 minutes**, using one plot as evidence.

| Line | What to write |
|---|---|
| **Location and window** | the example location and its planting window (start and end dates) |
| **First rain test passed** | the first day SPW v2's rain test passed inside the window (a date), or that it did not |
| **Evidence** | the weather evidence you would communicate, and the plot you will show |
| **Main uncertainty** | the main uncertainty, and the local checks you would add (soil, seed, local practice) |
| **If no recommendation** | what you would say if SPW gives no recommendation by the end of the window |

**How to save:** type your answers in the cell below and **run it** (Shift+Enter), then **run the cell after it** to check
and save them. After any change, run both cells again. The check only looks at the format; whether your answers hold up
is for the presentation.

In [ ]:
# ✍️ Type your answers after each colon (answers can run over several lines). Keep the labels.
# Then RUN THIS CELL, and run the next cell to check and save.
GROUP = ""   # your group name, e.g. "Group 3" (each group needs a different name)

ANSWERS = """
## Part A: key questions
Q1:
Q2:
Q3:
Q4:
Q5:

## Part B: planting window
Location and window:
First rain test passed:
Evidence:
Main uncertainty:
If no recommendation:
"""

In [ ]:
import datetime
import hashlib
import re

PARTS = ["Q1", "Q2", "Q3", "Q4", "Q5", "Location and window", "First rain test passed", "Evidence", "Main uncertainty",
         "If no recommendation"]
MIN_WORDS = {"Q1": 4, "Q2": 4, "Q3": 4, "Q4": 4, "Q5": 4, "Location and window": 3, "Evidence": 4,
             "Main uncertainty": 4, "If no recommendation": 4}
LABEL = re.compile(r"^\s*[-*]?\s*(" + "|".join(re.escape(p) for p in PARTS) + r")\s*:", re.I | re.M)
CANONICAL = {p.lower(): p for p in PARTS}
MONTH_NUMBER = {m: i for i, m in enumerate(["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}
MONTHS = r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?"
YEAR_OPT = r"(?:,?\s+(\d{4}))?"
DATE_PATTERNS = [  # (pattern, which groups are year, month, day)
    (r"\b(\d{4})-(\d{1,2})-(\d{1,2})\b", ("y", "m", "d")),
    (r"\b(\d{1,2})/(\d{1,2})(?:/(\d{2,4}))?\b", ("d", "m", "y")),
    (r"\b(\d{1,2})(?:st|nd|rd|th)?\s+" + MONTHS + YEAR_OPT, ("d", "M", "y")),
    (MONTHS + r"\s+(\d{1,2})(?:st|nd|rd|th)?\b" + YEAR_OPT, ("M", "d", "y"))]
NO_SIGNAL = re.compile(r"\b(none|never|did not pass|didn't pass|no (rain test|pass|signal|day))\b", re.I)


def count_dates(text):
    """Number of real, non-overlapping calendar dates in the text (2024-99-99, 31 Feb or 31 Apr do not count)."""
    spans = []
    for pattern, roles in DATE_PATTERNS:
        for m in re.finditer(pattern, text, re.I):
            parts = dict(zip(roles, m.groups()))
            try:
                month = MONTH_NUMBER[parts["M"][:3].lower()] if "M" in parts else int(parts["m"])
                year = int(parts["y"]) if parts.get("y") else 2024          # 2024: a leap year, so 29 Feb is allowed
                year = year + 2000 if year < 100 else year
                datetime.date(year, month, int(parts["d"]))
            except (ValueError, KeyError, TypeError):
                continue
            if not any(m.start() < end and start < m.end() for start, end in spans):
                spans.append((m.start(), m.end()))
    return len(spans)


def parse_answers(text):
    text = text.replace("\r\n", "\n")
    found, problems = {}, []
    marks = list(LABEL.finditer(text))
    for i, m in enumerate(marks):
        part = CANONICAL[m.group(1).lower()]
        end = marks[i + 1].start() if i + 1 < len(marks) else len(text)
        answer = re.sub(r"^\s*##.*$", "", text[m.end():end], flags=re.M).strip()
        if part in found:
            problems.append(f"'{part}' is written twice; only the first is checked.")
        else:
            found[part] = answer
    return found, problems


def check_part(part, answer):
    if answer is None:
        return "missing (label deleted)"
    if not answer:
        return "missing"
    words = len(answer.split())
    evidence = re.sub(r"\b(spw\s*)?v\d+(\.\d+)?\b", " ", answer, flags=re.I)   # "SPW v2" is not evidence
    if part.startswith("Q") and not re.search(r"\d", evidence):
        return "add a number or date from the notebook"
    if part == "Location and window" and count_dates(answer) < 2:
        return "give the window's start and end dates (e.g. 21 Nov to 7 Jan)"
    if part == "First rain test passed" and not (count_dates(answer) or NO_SIGNAL.search(answer)):
        return "give a date, or say that the rain test never passed"
    if words < MIN_WORDS.get(part, 1):
        return f"too short: {words} words (aim for {MIN_WORDS[part]}+)"
    return "ok"


if "ANSWERS" not in globals():
    raise RuntimeError("Run the answer cell above first (click it and press Shift+Enter), then run this cell.")

found, problems = parse_answers(ANSWERS)
status = pd.DataFrame([{"Part": "A: key questions" if p.startswith("Q") else "B: planting window", "Answer": p,
                        "Status": check_part(p, found.get(p))} for p in PARTS])
done = int(status.Status.eq("ok").sum())
display(results_table(status, f"{COUNTRY} group output: {done} of {len(status)} answers pass the format check",
                      label_colours={"A: key questions": "#367a9c", "B: planting window": "#2e8b57"}))

# file name: one per group; a name that loses characters when made safe gets a short code so groups cannot collide
name = GROUP.strip()
safe = re.sub(r"[^A-Za-z0-9_-]+", "_", name).strip("_")
if not name:
    safe = "unnamed_group"
elif safe != name:
    safe = (safe or "group") + "_" + hashlib.md5(name.encode("utf-8")).hexdigest()[:6]
path = Path(f"planting_window_{COUNTRY}_{YEAR}_{safe}.md").resolve()
action = "Updated" if path.exists() else "Saved"
document = (f"# Planting window: {COUNTRY} ({YEAR} season chosen in the notebook)\n\nGroup: {name or '(not given)'}\n\n"
            + ANSWERS.replace("\r\n", "\n").strip() + "\n")
path.write_text(document, encoding="utf-8")

todo = status.loc[status.Status.ne("ok")]
lines = [f"{action} <b>{path}</b>. Hand this file in, or use it for your 3-minute presentation."]
if not name:
    lines.append("⚠️ Set <b>GROUP</b> in the answer cell so your answers are not mixed up with another group's, then run both cells again.")
lines += [f"⚠️ {p}" for p in problems]
lines.append("Format checks passed for all ten answers. Whether they hold up is for the presentation."
             if todo.empty and not problems else
             "Fix the ⚠️ points above, then run both cells again." if todo.empty else
             f"Still to do: <b>{len(todo)}</b> answer(s), shown in the table above. Edit the answer cell, then run both cells again.")
key_result("Your group output", lines, colour="#2e8b57" if todo.empty and not problems and name else "#c77d1a")

### Extra (not on the task sheet): SPW test explorer

<details><summary><b>Open the explorer</b> (click to open; for groups who finish early)</summary>

Move the sliders and watch which days pass each test: the original rule (nearest of 20, 40 or 60 mm), the mean of the
members, the median, the probability, and the SPW v2 answer that combines the last two inside the planting window. If
the sliders do not appear, edit THRESHOLD_MM and P_CUTOFF in the cell below and run it again.

</details>

In [ ]:
# SPW test explorer — sliders if ipywidgets is installed, otherwise edit these values and rerun
# Country and year come from the choice at the top; the year can also be changed with the slider
THRESHOLD_MM = 30      # GAP uses 30 mm in all three countries
P_CUTOFF = 0.80        # GAP uses 0.80

SEASON_MONTHS = {"Zambia": ("09-01", "12-31"), "Malawi": ("09-01", "12-31"), "Nigeria": ("04-01", "07-31")}
_member_cache = {}

def daily_member_totals(country, year):
    """Spliced 6+5-day totals for every member on every advisory day (computed once per country and year)."""
    key = (country, year)
    if key not in _member_cache:
        use_country(country)
        support = pd.read_csv(DATA_DIR / f"{slug}_forecast_support.csv", parse_dates=["initialization_date", "date"])
        start, end = (pd.Timestamp(f"{year}-{d}") for d in SEASON_MONTHS[country])
        days = spw.set_index("date").loc[start:end]
        rows = []
        for day, record in days.iterrows():
            observed = observed_days(day - pd.Timedelta(days=V2_PAST_DAYS), day - pd.Timedelta(days=1)).sum()
            members = (support.loc[support.initialization_date.eq(pd.Timestamp(record.forecast_initialization_date))]
                       .set_index("date")[member_columns].reindex(pd.date_range(day, periods=V2_FUTURE_DAYS)))
            totals = observed + members.sum(axis=0).to_numpy(dtype=float) if members.notna().all().all() else None
            rows.append({"date": day, "totals": totals, "phase": spw_phase(day, country)})
        _member_cache[key] = (site, days, pd.DataFrame(rows).set_index("date"))
    return _member_cache[key]

def spw_test_explorer(country=COUNTRY, year=YEAR, threshold_mm=THRESHOLD_MM, p_cutoff=P_CUTOFF):
    site, original_flags, daily = daily_member_totals(country, year)
    ok = daily.totals.notna()
    mean = daily.totals[ok].apply(np.mean).reindex(daily.index)
    median = daily.totals[ok].apply(lambda t: np.percentile(t, 50)).reindex(daily.index)
    probability = daily.totals[ok].apply(
        lambda t: prob_exceed_normal(t.mean(), float(np.std(t)), threshold_mm)).reindex(daily.index)
    nearest = min([20, 40, 60], key=lambda t: abs(t - threshold_mm))
    in_window = daily.phase.eq("IN")
    passes_p = probability.ge(p_cutoff)
    passes_median = median.ge(threshold_mm)
    tests = {f"1 Original ({nearest} mm)": original_flags[f"spw_{nearest}"].eq(1),
             "2 Mean ≥ threshold": mean.ge(threshold_mm),
             "3 Median ≥ threshold": passes_median,
             f"4 P(total > threshold) ≥ {p_cutoff:.2f}": passes_p}
    decision = pd.Series("Outside window", index=daily.index)
    decision[in_window] = "Wait"
    decision[in_window & passes_median] = "Plant with caution"
    decision[in_window & passes_p] = "Rain test passed"
    decision[~ok] = "No forecast"

    fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True, gridspec_kw={"height_ratios": [1.5, 1]})
    ax = axes[0]
    labels = list(tests) + ["5 v2 decision"]
    for row, (name, passed) in enumerate(tests.items()):
        y = len(labels) - 1 - row
        ax.scatter(daily.index, np.repeat(y, len(daily)), color=np.where(passed, "#367a9c", "#eeeeee"), s=22)
    colours = {"Outside window": "#f5f5f5", "Wait": "#d98d64", "Plant with caution": "#f2c14e",
               "Rain test passed": "#367a9c", "No forecast": "#ffffff"}
    for state, colour in colours.items():
        days = decision.index[decision.eq(state)]
        if len(days):
            ax.scatter(days, np.repeat(0, len(days)), color=colour, s=22, label=state,
                       edgecolors="#bbbbbb" if state in ("Outside window", "No forecast") else "none")
    ax.set_yticks(range(len(labels)), labels[::-1])
    ax.set_ylim(-.7, len(labels) - .3)
    ax.legend(fontsize=8, ncol=5, loc="lower left", bbox_to_anchor=(0, -.22))
    ax.set_title(f"{country} | {site.site_name} | {year} | threshold {threshold_mm} mm, probability cut-off {p_cutoff:.2f}")
    ax = axes[1]
    ax.plot(probability.index, probability, "o-", ms=3, lw=1, color="#367a9c", label="Probability of reaching the threshold")
    ax.axhline(p_cutoff, color="black", ls="--", lw=1, label="Cut-off")
    if in_window.any():
        ax.axvspan(daily.index[in_window].min(), daily.index[in_window].max(), color="#d9e7cf", alpha=.45, zorder=0,
                   label="Planting window")
    ax.set_ylim(-.03, 1.03); ax.set_ylabel("Probability"); ax.grid(axis="y", alpha=.2)
    ax.legend(fontsize=8, loc="upper left")
    fig.autofmt_xdate(); plt.tight_layout(); plt.show()

    def first(mask):
        return mask.index[mask].min().date() if mask.any() else "None"
    summary = [{"Test": name, "Days passed": int(passed.sum()), "First day": first(passed)} for name, passed in tests.items()]
    for state in ["Rain test passed", "Plant with caution"]:
        summary.append({"Test": f"5 v2: {state.lower()}", "Days passed": int(decision.eq(state).sum()),
                        "First day": first(decision.eq(state))})
    display(results_table(pd.DataFrame(summary), "Days each test was passed", bar_column="Days passed"))

try:
    import ipywidgets as widgets
    widgets.interact(spw_test_explorer,
                     country=widgets.fixed(COUNTRY),
                     year=widgets.Dropdown(options=[2023, 2024], value=YEAR, description="Year"),
                     threshold_mm=widgets.IntSlider(value=THRESHOLD_MM, min=10, max=60, step=5, description="Threshold mm",
                                                    continuous_update=False),
                     p_cutoff=widgets.FloatSlider(value=P_CUTOFF, min=.5, max=.95, step=.05, description="Cut-off",
                                                  continuous_update=False))
except ImportError:
    print("ipywidgets is not installed: showing the values set above. Edit them and rerun this cell.")
    spw_test_explorer()